<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# 🏥 DataProjectLab — Projet Santé
## Prévision de la demande en médicaments
### Notebook 2 — Analyse SQL & Exploration Python (EDA)
---
**Prérequis :** Avoir complété le Notebook 1 et les Tâches 0 et 1.

## PARTIE A — ANALYSE SQL

Dans cette partie, tu vas interroger les données avec SQL pour répondre aux premières questions business.
Tu peux utiliser **DuckDB** (recommandé, s'installe avec `pip install duckdb`) ou charger les CSV dans une base SQLite / PostgreSQL locale.

In [ ]:
# Installation et connexion DuckDB
# pip install duckdb pandas matplotlib seaborn

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

import os, sys, warnings
warnings.filterwarnings('ignore')

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/DataProjectLab/projects/pharmacy_analytics/"
else:
    SAVE_PATH = "./outputs/"
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"📁 Environnement : {'Colab' if IN_COLAB else 'Local'}")
print(f"📁 Dossier       : {SAVE_PATH}")

# ── Chemins des fichiers ────────────────────────────────────────────────────
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/data/"
clean_path = f"{SAVE_PATH}pharmacy_clean_analytics.csv"


# Connexion et chargement des CSV
con = duckdb.connect()

con.execute(f"""
    CREATE TABLE medicaments    AS SELECT * FROM read_csv_auto('{BASE_URL}medicaments.csv');
    CREATE TABLE services       AS SELECT * FROM read_csv_auto('{BASE_URL}services.csv');
    CREATE TABLE fournisseurs   AS SELECT * FROM read_csv_auto('{BASE_URL}fournisseurs.csv');
    CREATE TABLE consommations  AS SELECT * FROM read_csv_auto('{BASE_URL}consommations.csv');
    CREATE TABLE commandes      AS SELECT * FROM read_csv_auto('{BASE_URL}commandes_fournisseurs.csv');
    CREATE TABLE ruptures       AS SELECT * FROM read_csv_auto('{BASE_URL}ruptures_stock.csv');
""")

print("✅ 6 tables chargées avec succès")
print(con.execute("SELECT COUNT(*) FROM consommations").fetchone()[0], "lignes dans consommations")


📁 Environnement : Local
📁 Dossier       : ./outputs/
✅ 6 tables chargées avec succès
53808 lignes dans consommations


In [ ]:
# Lier la connexion à JupySQL pour les cellules %%sql
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print("%%sql prêt ✅")

---
### 🔷 SQL — Tâche 1 : Vue d'ensemble des consommations

**Objectif :** Comprendre la volumétrie globale et les grandes tendances.

**Questions à répondre avec tes requêtes :**
- Quelle est la consommation totale (en unités et en euros) sur toute la période ?
- Quel est le top 5 des médicaments les plus consommés en quantité ?
- Quel est le top 5 en valeur (coût total) ?

> 💡 **Hint :** Pense à joindre `consommations` avec `medicaments` pour avoir les noms. Utilise `SUM`, `GROUP BY` et `ORDER BY DESC`.

In [ ]:
# ✏️ TÂCHE 1 — Requêtes vue d'ensemble
# Écris tes 3 requêtes SQL ci-dessous

# Requête 1 : Totaux généraux
%%sql 
    SELECT
        -- Votre code ici
    FROM tickets


---
### 🔷 SQL — Tâche 2 : Analyse temporelle

**Objectif :** Identifier les tendances et saisonnalités dans les consommations.

**Questions :**
- Quelle est la consommation mensuelle totale (en unités) ? Vois-tu une tendance ?
- Quels mois ont la consommation la plus élevée ? La plus faible ?
- Y a-t-il une différence entre jours de semaine et week-end ?

> 💡 **Hint :** Utilise `DATE_TRUNC('month', date)` pour agréger par mois. Pour les jours de la semaine : `DAYOFWEEK(date)`.

In [ ]:
# ✏️ TÂCHE 2 — Analyse temporelle

# Consommation mensuelle
%%sql df_mensuel <<
    SELECT
        -- Votre code ici
    FROM tickets

# Visualisation (complète après avoir écrit ta requête)
# fig, ax = plt.subplots(figsize=(14, 5))
# ax.plot(df_mensuel['mois'], df_mensuel['total_unites'])
# plt.title('Consommation mensuelle totale')
# plt.show()


---
### 🔷 SQL — Tâche 3 : Consommation par service

**Objectif :** Identifier quels services sont les plus consommateurs et pour quels médicaments.

**Questions :**
- Quel service consomme le plus (en coût total) ?
- Pour chaque service, quel est son médicament numéro 1 ?
- Calcule le coût moyen par lit par jour pour chaque service

> 💡 **Hint :** Pour "le médicament numéro 1 par service", utilise une window function : `ROW_NUMBER() OVER (PARTITION BY id_service ORDER BY SUM(quantite_consommee) DESC)`.

In [ ]:
# ✏️ TÂCHE 3 — Analyse par service

# Top médicament par service (window function)
%%sql df_srv_med <<
    SELECT
        -- Votre code ici
    FROM tickets


---
### 🔷 SQL — Tâche 4 : Analyse des ruptures et fournisseurs

**Objectif :** Comprendre les défaillances passées et évaluer les fournisseurs.

**Questions :**
- Quels médicaments ont eu le plus de ruptures ? Combien de patients affectés au total ?
- Quel est le taux de livraison à temps par fournisseur ?
- Quel fournisseur a le meilleur taux de service pour les médicaments critiques ?

> 💡 **Hint :** Pour le taux de livraison à temps : `SUM(CASE WHEN retard_jours = 0 THEN 1 ELSE 0 END) / COUNT(*)`.

In [ ]:
# ✏️ TÂCHE 4 — Ruptures et fournisseurs

# Performance fournisseurs
%%sql df_frn_perf <<
    SELECT
        -- Votre code ici
    FROM tickets


---
## PARTIE B — EXPLORATION PYTHON (EDA)

Maintenant que tu as une vision SQL des données, tu vas approfondir l'analyse avec Python pour préparer la modélisation.

In [7]:
# Chargement des données pour Python
df_conso  = pd.read_csv(f'{BASE_URL}consommations.csv', parse_dates=['date'])
df_med    = pd.read_csv(f'{BASE_URL}medicaments.csv')
df_srv    = pd.read_csv(f'{BASE_URL}services.csv')
df_cmd    = pd.read_csv(f'{BASE_URL}commandes_fournisseurs.csv', parse_dates=['date_commande','date_livraison_reelle'])
df_rup    = pd.read_csv(f'{BASE_URL}ruptures_stock.csv', parse_dates=['date_debut','date_fin'])
df_frn    = pd.read_csv(f'{BASE_URL}fournisseurs.csv')

# Fusion principale
df = df_conso.merge(df_med[['id_medicament','nom','categorie','medicament_critique']], on='id_medicament')
df = df.merge(df_srv[['id_service','nom_service']], on='id_service')

print(f"Dataset fusionné : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head()


Dataset fusionné : 53,808 lignes × 10 colonnes


,id_consommation,date,id_medicament,id_service,quantite_consommee,cout_euro,nom,categorie,medicament_critique,nom_service
0,CONSO000001,2022-01-01,MED001,SRV001,8,20.0,Amoxicilline 500mg,Antibiotique,True,Urgences
1,CONSO000002,2022-01-01,MED002,SRV001,13,10.4,Paracétamol 1g,Analgésique,True,Urgences
2,CONSO000003,2022-01-01,MED003,SRV001,10,12.0,Ibuprofène 400mg,Anti-inflammatoire,True,Urgences
3,CONSO000004,2022-01-01,MED011,SRV001,6,72.0,Ceftriaxone 1g,Antibiotique,True,Urgences
4,CONSO000005,2022-01-01,MED013,SRV001,4,27.2,Dexaméthasone 4mg,Corticoïde,True,Urgences


---
### 🔶 Python — Tâche 5 : Audit qualité des données

**Objectif :** Vérifier la qualité avant toute analyse.

**À vérifier :**
- Valeurs nulles par colonne
- Doublons éventuels
- Cohérence des dates (pas de dates futures, pas de gaps)
- Distribution des quantités (outliers ?)

> 💡 **Hint :** `df.isnull().sum()`, `df.duplicated().sum()`, `df.describe()`, `df['quantite_consommee'].plot(kind='box')`

In [ ]:
# ✏️ TÂCHE 5 — Audit qualité

# 1. Valeurs nulles
# 👉 ÉCRIS TON CODE ICI

# 2. Doublons
# 👉 ÉCRIS TON CODE ICI

# 3. Distribution des quantités (boxplot)
# 👉 ÉCRIS TON CODE ICI


---
### 🔶 Python — Tâche 6 : Visualisations analytiques

**Objectif :** Produire 4 visualisations qui racontent l'histoire des données.

**Visualisation 1 :** Évolution mensuelle de la consommation totale + ligne de tendance
**Visualisation 2 :** Heatmap consommation par médicament × mois (top 10 médicaments)
**Visualisation 3 :** Comparaison week/week-end pour les médicaments critiques
**Visualisation 4 :** Distribution des retards fournisseurs (histogram)

> 💡 **Hint :** Pour la ligne de tendance : `numpy.polyfit` ou `scipy.stats.linregress`. Pour la heatmap : `df.pivot_table` puis `sns.heatmap`.

In [ ]:
# ✏️ TÂCHE 6 — Visualisation 1 : Évolution mensuelle
fig, ax = plt.subplots(figsize=(14, 5))

# Agrège par mois
# df_monthly = df.groupby(...)[...].sum().reset_index()

# 👉 ÉCRIS TON CODE ICI

plt.title('Consommation mensuelle totale en unités', fontsize=14, pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# ✏️ TÂCHE 6 — Visualisation 2 : Heatmap médicament × mois

# 👉 ÉCRIS TON CODE ICI (pivot_table + heatmap)


---
### 🔶 Python — Tâche 7 : Feature Engineering pour le ML

**Objectif :** Créer les variables qui serviront à entraîner le modèle de prévision.

**Variables à créer pour chaque ligne (médicament × date) :**
- `consommation_7j` — moyenne mobile sur 7 jours
- `consommation_30j` — moyenne mobile sur 30 jours
- `lag_7` — consommation d'il y a 7 jours (même jour semaine précédente)
- `lag_28` — consommation d'il y a 28 jours
- `mois` — numéro du mois (1-12) pour la saisonnalité
- `jour_semaine` — 0=Lundi, 6=Dimanche
- `est_weekend` — variable booléenne
- `semaine_annee` — numéro de semaine (1-52)

> ⚠️ **Important :** Fais ce feature engineering **par médicament séparément** pour éviter les fuites de données entre médicaments.

In [ ]:
# ✏️ TÂCHE 7 — Feature Engineering
# Structure suggérée :

def create_features(df_med_single):
    """
    Crée les features temporelles pour un seul médicament.
    df_med_single : DataFrame filtré sur 1 médicament, trié par date
    """
    df_f = df_med_single.copy().sort_values('date')
    
    # 👉 CRÉE LES FEATURES ICI
    # df_f['consommation_7j']  = ...
    # df_f['consommation_30j'] = ...
    # df_f['lag_7']  = ...
    # df_f['lag_28'] = ...
    # df_f['mois']   = ...
    # ...
    
    return df_f

# Applique à tous les médicaments
# df_features = df.groupby('id_medicament', group_keys=False).apply(create_features)
# df_features = df_features.dropna()  # supprimer les lignes avec NaN issus des lags
# print(f"Dataset features : {df_features.shape}")
#df_features.to_csv(clean_path, index=False)


---
## ✅ Checklist Notebook 2

Avant de passer au Notebook ML, vérifie :

- [ ] Tâche 1 — 3 requêtes SQL vue d'ensemble exécutées
- [ ] Tâche 2 — Analyse temporelle SQL + graphique mensuel
- [ ] Tâche 3 — Window function par service réussie
- [ ] Tâche 4 — Performance fournisseurs calculée
- [ ] Tâche 5 — Audit qualité données Python
- [ ] Tâche 6 — 4 visualisations produites
- [ ] Tâche 7 — Feature engineering complet, 0 NaN restant

> ✅ Toutes les cases cochées ? Passe au **Notebook 3 — Machine Learning**